# Trabalho Prático 2 — Redes Neurais (CIFAR-10)

- **Disciplina:** Sistemas Inteligentes — UFSC
- **Professora:** Nathalia da Cruz Alves
- **Equipe:** `[preencher: Nome(s) e matrícula(s)]`
- **Data:** junho de 2026

## Objetivo
Classificar imagens do conjunto **CIFAR-10** (32×32 RGB, 10 classes) com redes neurais artificiais,
percorrendo quatro experimentos, **nesta ordem**:

1. **MLP simples** (sem camadas convolucionais);
2. **CNN com aumento de dados** (*data augmentation*);
3. **Busca de hiperparâmetros** com tabela comparativa;
4. **Avaliação final** do melhor modelo no conjunto de teste.

Cada parte traz os gráficos de *loss* e *acurácia* por época (treino × validação) e a respectiva
análise escrita.

> **Execução (Google Colab):** ative a GPU em *Ambiente de execução → Alterar o tipo de ambiente de
> execução → GPU* e use *Ambiente de execução → Executar tudo*. A execução completa leva ~20–35 min.
> Ao final, salve com *Arquivo → Fazer download → Fazer o download do .ipynb* para entregar o
> notebook **com as saídas**.

## Referências e uso de ferramentas
- **TensorFlow / Keras** — biblioteca de redes neurais (documentação oficial).
- **CIFAR-10** — Krizhevsky, A. (2009). *Learning Multiple Layers of Features from Tiny Images*.
- **Ferramenta de IA generativa:** *Claude (Anthropic)*, via *Claude Code*, utilizada para **auxiliar
  na estruturação do notebook, na geração do código TensorFlow/Keras e na redação das análises**. Todo
  o conteúdo foi **revisado e validado pela equipe**.

## Configuração, sementes e checagem de GPU

In [ ]:
import os, time, random, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reprodutibilidade
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs disponíveis:", gpus if gpus else "NENHUMA — ative Ambiente de execução > GPU no Colab")

In [ ]:
# Hiperparâmetros globais de execução (reduza os valores para rodar mais rápido)
EPOCHS_MLP    = 25      # Parte 1
EPOCHS_CNN    = 30      # Parte 2
EPOCHS_SEARCH = 12      # Parte 3 (por combinação)
EPOCHS_FINAL  = 30      # Parte 4
BATCH_SIZE    = 128
VAL_SIZE      = 5000    # exemplos separados do treino para validação
SEARCH_SUBSET = 20000   # subconjunto do treino usado na busca (acelera a Parte 3)
N_COMBOS      = 8       # nº de combinações de hiperparâmetros amostradas na busca

## Carregamento e preparação dos dados

Carregamos o CIFAR-10, normalizamos os pixels para o intervalo `[0, 1]` e separamos um conjunto de
**validação** a partir do treino (de forma **estratificada**, mantendo as classes balanceadas). O
conjunto de **teste fica intocado até a Parte 4**.

In [ ]:
from sklearn.model_selection import train_test_split

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normaliza para [0, 1] e usa rótulos inteiros 1D (-> sparse_categorical_crossentropy)
x_train_full = x_train_full.astype("float32") / 255.0
x_test       = x_test.astype("float32") / 255.0
y_train_full = y_train_full.astype("int64").reshape(-1)
y_test       = y_test.astype("int64").reshape(-1)

# Separa validação do treino (estratificado)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=VAL_SIZE, stratify=y_train_full, random_state=SEED)

classes_pt = ["avião", "automóvel", "pássaro", "gato", "cervo",
              "cachorro", "sapo", "cavalo", "navio", "caminhão"]

print("Treino:   ", x_train.shape, y_train.shape)
print("Validação:", x_val.shape, y_val.shape)
print("Teste:    ", x_test.shape, y_test.shape)
print("Classes:  ", classes_pt)

In [ ]:
# Amostra de imagens do conjunto de treino
plt.figure(figsize=(10, 4))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(classes_pt[y_train[i]], fontsize=9)
    plt.axis("off")
plt.suptitle("Amostras do CIFAR-10 (treino)")
plt.tight_layout()
plt.show()

## Utilitários (gráficos e aumento de dados)

`plot_history` desenha *loss* e *acurácia* (treino × validação) por época — reutilizado nas Partes 1,
2 e 4. `make_augmentation` cria o bloco de *data augmentation* usado nas Partes 2 a 4.

In [ ]:
def plot_history(history, titulo=""):
    """Plota loss e acurácia (treino x validação) por época, lado a lado."""
    h = history.history
    acc_key = "accuracy" if "accuracy" in h else "acc"
    epochs = range(1, len(h["loss"]) + 1)

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].plot(epochs, h["loss"], "o-", label="treino")
    ax[0].plot(epochs, h["val_loss"], "s-", label="validação")
    ax[0].set_title(f"Loss por época — {titulo}")
    ax[0].set_xlabel("época"); ax[0].set_ylabel("loss (entropia cruzada)")
    ax[0].legend(); ax[0].grid(True, alpha=0.3)

    ax[1].plot(epochs, h[acc_key], "o-", label="treino")
    ax[1].plot(epochs, h["val_" + acc_key], "s-", label="validação")
    ax[1].set_title(f"Acurácia por época — {titulo}")
    ax[1].set_xlabel("época"); ax[1].set_ylabel("acurácia")
    ax[1].legend(); ax[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


def resumo_metricas(history, nome):
    """Imprime as métricas finais de treino e validação."""
    h = history.history
    acc_key = "accuracy" if "accuracy" in h else "acc"
    print(f"[{nome}] métricas da última época")
    print(f"  loss treino    = {h['loss'][-1]:.4f}   | acurácia treino    = {h[acc_key][-1]:.4f}")
    print(f"  loss validação = {h['val_loss'][-1]:.4f}   | acurácia validação = {h['val_'+acc_key][-1]:.4f}")
    print(f"  gap de acurácia (treino - validação) = {h[acc_key][-1] - h['val_'+acc_key][-1]:.4f}")


def make_augmentation():
    """Bloco de aumento de dados (ativo apenas durante o treino)."""
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.1),
        layers.RandomTranslation(0.1, 0.1),
    ], name="data_augmentation")

# Parte 1 — Rede neural MLP simples

Uma **MLP (Multi-Layer Perceptron)** é uma rede totalmente conectada: cada imagem 32×32×3 é
"achatada" em um vetor de 3.072 valores e processada por camadas densas. **Não há camadas
convolucionais** — portanto o modelo **não explora a estrutura espacial** (vizinhança de pixels) das
imagens. Serve como linha de base.

In [ ]:
mlp = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Flatten(),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax"),
], name="MLP_simples")

mlp.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
mlp.summary()

# Sem EarlyStopping de propósito: queremos a curva completa por época.
hist_mlp = mlp.fit(x_train, y_train,
                   validation_data=(x_val, y_val),
                   epochs=EPOCHS_MLP, batch_size=BATCH_SIZE, verbose=2)

In [ ]:
plot_history(hist_mlp, "MLP simples")
resumo_metricas(hist_mlp, "MLP")

## Análise — MLP simples

**Como a *loss* e a *acurácia* são calculadas.** A *loss* é a **entropia cruzada categórica**: para
cada exemplo, a rede produz uma distribuição de probabilidade sobre as 10 classes (via *softmax*) e a
perda é `-log(p)` da classe correta; a *loss* reportada é a **média** desse valor sobre os exemplos.
É uma quantidade **contínua** que mede o quão *confiante e correta* a rede está. A **acurácia** é a
**fração de exemplos** em que a classe de maior probabilidade (`argmax`) coincide com o rótulo
verdadeiro — uma métrica **discreta** (acertou/errou), que ignora a confiança.

**Por que as duas não são proporcionais.** A acurácia só muda quando o `argmax` muda; a *loss* muda
continuamente com a *probabilidade* atribuída à classe correta. Assim, o modelo pode **reduzir a
loss sem ganhar acurácia** (fica mais confiante em acertos que já fazia) ou, o caso mais ilustrativo,
**piorar a `val_loss` enquanto a `val_accuracy` fica estável** — sinal clássico de *overfitting*: o
modelo passa a errar com **muita confiança**, o que penaliza fortemente a entropia cruzada (cada erro
confiante contribui com `-log(p)` grande), embora cada erro conte igual na acurácia. Por isso é comum
ver a curva de `val_loss` subir enquanto a `val_accuracy` quase não se move.

**Desempenho e generalização.** A MLP atinge tipicamente **~45–52% de acurácia de validação** no
CIFAR-10. Há uma **distância (*gap*)** entre treino e validação — a *loss* de treino continua caindo
enquanto a de validação estaciona/sobe —, indicando **capacidade limitada de generalização**. A causa
principal é o **achatamento** da imagem, que descarta a informação espacial. Isso **motiva o uso de
redes convolucionais** na Parte 2.

# Parte 2 — Rede convolucional (CNN) com aumento de dados

As **camadas convolucionais** aplicam filtros locais que percorrem a imagem, explorando a
**vizinhança de pixels** e o **compartilhamento de pesos** — bem mais adequadas a imagens do que
camadas densas. Acrescentamos **aumento de dados (*data augmentation*)**: transformações aleatórias
(espelhamento, rotação, zoom, translação) aplicadas **somente no treino**, gerando variações novas a
cada época.

In [ ]:
data_augmentation = make_augmentation()

# Visualiza o efeito do aumento de dados sobre uma única imagem
plt.figure(figsize=(10, 4))
img = x_train[7:8]
for i in range(10):
    aug = data_augmentation(img, training=True)
    plt.subplot(2, 5, i + 1)
    plt.imshow(np.clip(aug[0].numpy(), 0, 1))
    plt.axis("off")
plt.suptitle(f"Aumento de dados aplicado a uma imagem ('{classes_pt[y_train[7]]}')")
plt.tight_layout(); plt.show()

In [ ]:
cnn = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    data_augmentation,

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.30),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.40),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
], name="CNN_com_augmentation")

cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"])
cnn.summary()

hist_cnn = cnn.fit(x_train, y_train,
                   validation_data=(x_val, y_val),
                   epochs=EPOCHS_CNN, batch_size=BATCH_SIZE, verbose=2)

In [ ]:
plot_history(hist_cnn, "CNN + data augmentation")
resumo_metricas(hist_cnn, "CNN")

## Análise — CNN com aumento de dados

**O que o aumento de dados indica sobre a quantidade de dados por categoria.** O CIFAR-10 tem
**5.000 imagens por classe** no treino — pouco diante da variedade de poses, fundos e iluminações
reais. O *data augmentation* **gera variações plausíveis** das imagens existentes, **aumentando
artificialmente o número efetivo de exemplos por categoria**. Ou seja, funciona como um **substituto
barato a "coletar mais dados"** e atua como **regularizador**: a rede vê cada objeto sob muitas
transformações e é forçada a aprender características **invariantes** (um cavalo continua sendo um
cavalo espelhado/rotacionado) em vez de **decorar** exemplos específicos. A implicação é direta:
**quanto menos dados por categoria, mais o aumento de dados ajuda**.

**Desempenho e generalização.** A CNN com aumento de dados alcança tipicamente **~75–82% de acurácia
de validação**, muito acima da MLP. O **gap treino/validação fica menor**, indicando **melhor
generalização**. É comum, no início do treino, a **acurácia de treino ficar *abaixo* da de
validação**: como o aumento de dados só age no treino, os lotes de treino são "mais difíceis" que os
de validação (sem transformação) — efeito **esperado**, não um erro.

# Parte 3 — Busca de hiperparâmetros

Implementamos um procedimento que testa **diferentes combinações de hiperparâmetros** e organiza os
resultados em uma **tabela comparativa**. Para manter a busca **leve** (executável em uma única
sessão de Colab), usamos um **backbone CNN compacto** (2 blocos convolucionais), **sem aumento de
dados** e treinado por poucas épocas com *EarlyStopping*, sobre um **subconjunto do treino**. A busca
serve como um **proxy barato** para *ranquear* as combinações; a vencedora é treinada por completo (e
com aumento de dados) na Parte 4.

**Espaço de busca:**

- **learning rate** ∈ {1e-2, 1e-3, 5e-4}
- **neurônios da camada densa** ∈ {128, 256}
- **dropout** ∈ {0.3, 0.5}

In [ ]:
def build_cnn(dense_units, dropout, learning_rate, augment=False):
    """CNN compacta e parametrizável usada na busca (Parte 3) e no modelo final (Parte 4)."""
    inputs = keras.Input(shape=(32, 32, 3))
    x = make_augmentation()(inputs) if augment else inputs
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
espaco = {
    "learning_rate": [1e-2, 1e-3, 5e-4],
    "dense_units":   [128, 256],
    "dropout":       [0.3, 0.5],
}
grade = [dict(zip(espaco, v)) for v in itertools.product(*espaco.values())]  # 12 combinações
random.seed(SEED)
combos = random.sample(grade, N_COMBOS)  # amostra leve

# Subconjunto do treino para acelerar a busca
xs, ys = x_train[:SEARCH_SUBSET], y_train[:SEARCH_SUBSET]

resultados = []
for i, hp in enumerate(combos, 1):
    print(f"=== Combinação {i}/{len(combos)}: {hp} ===")
    keras.backend.clear_session()
    model = build_cnn(**hp, augment=False)
    early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                          restore_best_weights=True)
    t0 = time.time()
    h = model.fit(xs, ys, validation_data=(x_val, y_val),
                  epochs=EPOCHS_SEARCH, batch_size=BATCH_SIZE,
                  callbacks=[early], verbose=0)
    dt = time.time() - t0
    val_acc  = float(np.max(h.history["val_accuracy"]))
    val_loss = float(np.min(h.history["val_loss"]))
    resultados.append({**hp, "val_acc": val_acc, "val_loss": val_loss,
                       "épocas": len(h.history["loss"]), "tempo_s": round(dt, 1)})
    print(f"    -> val_acc={val_acc:.4f} | val_loss={val_loss:.4f} | {dt:.1f}s")

In [ ]:
# Tabela comparativa, ordenada da melhor para a pior combinação
tabela = pd.DataFrame(resultados).sort_values("val_acc", ascending=False).reset_index(drop=True)
tabela

In [ ]:
melhor = tabela.iloc[0]
melhores_hp = {
    "dense_units":   int(melhor["dense_units"]),
    "dropout":       float(melhor["dropout"]),
    "learning_rate": float(melhor["learning_rate"]),
}
print("Melhores hiperparâmetros encontrados:", melhores_hp)
print(f"val_acc = {melhor['val_acc']:.4f} | val_loss = {melhor['val_loss']:.4f}")

## Análise — quais hiperparâmetros mais influenciaram

Observando a tabela comparativa (ordenada por `val_acc`):

- **Learning rate** costuma ser o fator **mais determinante**. Um valor alto demais (**1e-2**) pode
  deixar o treino **instável** ou estagnado; valores menores (**1e-3 / 5e-4**) convergem de forma mais
  estável e tendem a liderar a tabela. É o hiperparâmetro que mais altera o resultado porque controla
  **diretamente o tamanho do passo** da otimização.
- **Número de neurônios** da camada densa (**128 → 256**) tem efeito **secundário**: mais neurônios
  aumentam a capacidade do modelo, com ganho geralmente pequeno e algum risco de *overfitting* se a
  regularização não acompanhar.
- **Dropout** (**0.3 vs 0.5**) atua na **regularização**: pouco impacto na acurácia bruta em poucas
  épocas, mas valores maiores tendem a **reduzir o gap treino/validação**.

> Os **melhores valores encontrados** são exibidos pela célula de código acima (`melhores_hp`) e
> reaproveitados na Parte 4. *(Confirme os números conforme a sua execução: pequenas variações entre
> sessões são normais.)*

# Parte 4 — Avaliação final no conjunto de teste

Reconstruímos o modelo com a **melhor combinação de hiperparâmetros** da Parte 3, agora treinado de
forma **completa e com aumento de dados** (mais épocas, *EarlyStopping* e redução do *learning rate*
em platôs). Só então usamos o **conjunto de teste**, mantido **intocado** até aqui, para estimar o
desempenho em **dados não vistos**.

In [ ]:
keras.backend.clear_session()
modelo_final = build_cnn(**melhores_hp, augment=True)
modelo_final.summary()

cbs = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
]
hist_final = modelo_final.fit(x_train, y_train,
                              validation_data=(x_val, y_val),
                              epochs=EPOCHS_FINAL, batch_size=BATCH_SIZE,
                              callbacks=cbs, verbose=2)
plot_history(hist_final, "Modelo final (melhores hiperparâmetros)")

In [ ]:
test_loss, test_acc = modelo_final.evaluate(x_test, y_test, verbose=0)
print(f"CONJUNTO DE TESTE  ->  loss = {test_loss:.4f} | acurácia = {test_acc:.4f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = np.argmax(modelo_final.predict(x_test, verbose=0), axis=1)
print(classification_report(y_test, y_pred, target_names=classes_pt, digits=3))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=classes_pt).plot(
    ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Matriz de confusão — conjunto de teste")
plt.tight_layout(); plt.show()

In [ ]:
# Exemplos de predições no teste (verde = correto, vermelho = erro)
plt.figure(figsize=(12, 5))
idx = np.random.RandomState(SEED).choice(len(x_test), 15, replace=False)
for plot_i, i in enumerate(idx):
    plt.subplot(3, 5, plot_i + 1)
    plt.imshow(x_test[i])
    cor = "green" if y_pred[i] == y_test[i] else "red"
    plt.title(f"prev: {classes_pt[y_pred[i]]}\nreal: {classes_pt[y_test[i]]}", color=cor, fontsize=8)
    plt.axis("off")
plt.suptitle("Predições no conjunto de teste")
plt.tight_layout(); plt.show()

## Análise — desempenho final e generalização

- **Métricas de teste.** A acurácia e a *loss* de teste (impressas acima) resumem o desempenho em
  dados **nunca vistos** no treino. A **proximidade entre validação e teste** indica que a seleção de
  modelo **não "vazou" informação** do conjunto de teste — o processo é honesto e o resultado é
  confiável.
- **Matriz de confusão.** As classes de **animais** (gato, cachorro, cervo, pássaro) tendem a ser as
  mais confundidas entre si — têm formas e texturas parecidas em 32×32 —, enquanto **veículos**
  (avião, navio, caminhão, automóvel) costumam ser mais fáceis. O `classification_report` por classe
  detalha *precision* e *recall* individuais.
- **Conclusão sobre generalização.** O *pipeline* (CNN + aumento de dados + hiperparâmetros ajustados)
  **generaliza bem** para dados não vistos, com desempenho muito superior à MLP da Parte 1. Ganhos
  adicionais viriam de redes mais profundas (p. ex. *ResNet*), *transfer learning* ou treino por mais
  épocas.

# Conclusão

Percorremos quatro experimentos no CIFAR-10: (1) uma **MLP** como linha de base, limitada por ignorar
a estrutura espacial; (2) uma **CNN com aumento de dados**, que melhora muito a acurácia e a
generalização; (3) uma **busca de hiperparâmetros** com tabela comparativa, identificando os fatores
mais influentes (com destaque para o *learning rate*); e (4) a **avaliação final no teste**, que
confirma a boa capacidade de generalização do melhor modelo.

**Referências:** TensorFlow/Keras (documentação oficial); Krizhevsky, A. (2009), *Learning Multiple
Layers of Features from Tiny Images*. **Uso de IA:** *Claude (Anthropic)* via *Claude Code* para
auxiliar na estruturação, codificação e redação — revisado pela equipe.